# 60. Qt Desktop Apps — `pyside6 (PyQt)`
**Python Libraries Lab · Phase 3 · GUI, Games & Multimedia**

_Professional desktop apps with the Qt toolkit._

⚙️ **Runs in Colab** after the setup cell below.

Qt is a professional toolkit used for apps like VLC, Autodesk Maya and many tools in cars and factories. PySide6 and PyQt6 bring Qt to Python. Compared with Tkinter, Qt gives you modern looking widgets, tables, menus, docking panels and charts, and it scales up to large applications.

**In this notebook:** You will build a ShopEasy inventory window with a search box, a product table, a low stock filter and a status bar. In Colab it runs offscreen, so you will test it by simulating typing and clicking, and save a screenshot. On your laptop it opens as a real window.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
Qt opens real windows on your laptop. In Colab we run Qt **offscreen**: the app is built and screenshotted as an image so you can see it. PySide6 is the official Qt for Python; PyQt6 is almost identical.

The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q pyside6
import importlib.metadata as md
for pkg in ['pyside6']:
    print(pkg, md.version(pkg))

In [ ]:
# Qt needs a few system libraries in Colab, and runs "offscreen" (no visible window)
!apt-get -qq install -y libegl1 libxkbcommon0 libgl1 libfontconfig1 libdbus-1-3 > /dev/null 2>&1
import os
os.environ["QT_QPA_PLATFORM"] = "offscreen"
print("Qt ready in offscreen mode")

## Step 1 · The inventory window
Saved as a file so you can also run it on your laptop. Signals (`textChanged`, `toggled`) connect to the `refresh` slot.

In [ ]:
%%writefile qt_inventory.py
import sys
from PySide6.QtWidgets import (QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout, QLineEdit,
                               QCheckBox, QTableWidget, QTableWidgetItem, QHeaderView, QLabel)
from PySide6.QtGui import QColor

PRODUCTS = [("Steel Bottle", "Kitchen", 499, 40), ("Yoga Mat", "Fitness", 1299, 3), ("Desk Lamp", "Home", 899, 7),
            ("Notebook", "Stationery", 149, 200), ("Headphones", "Electronics", 2499, 2), ("Dumbbells", "Fitness", 1999, 15)]

class Inventory(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("ShopEasy inventory")
        self.resize(620, 360)
        self.search = QLineEdit(placeholderText="Search products...")
        self.low = QCheckBox("Low stock only (< 10)")
        self.table = QTableWidget(0, 4)
        self.table.setHorizontalHeaderLabels(["Product", "Category", "Price", "Stock"])
        self.table.horizontalHeader().setSectionResizeMode(QHeaderView.Stretch)
        top = QHBoxLayout(); top.addWidget(self.search); top.addWidget(self.low)
        box = QVBoxLayout(); box.addLayout(top); box.addWidget(self.table)
        central = QWidget(); central.setLayout(box); self.setCentralWidget(central)
        self.search.textChanged.connect(self.refresh)     # signal -> slot
        self.low.toggled.connect(self.refresh)
        self.refresh()

    def visible_products(self):
        q = self.search.text().lower()
        return [p for p in PRODUCTS if q in p[0].lower() and (not self.low.isChecked() or p[3] < 10)]

    def refresh(self):
        rows = self.visible_products()
        self.table.setRowCount(len(rows))
        for r, (name, cat, price, stock) in enumerate(rows):
            for c, val in enumerate([name, cat, f"Rs. {price:,}", str(stock)]):
                item = QTableWidgetItem(val)
                if stock < 10:
                    item.setBackground(QColor("#FFE3E3"))
                self.table.setItem(r, c, item)
        self.statusBar().showMessage(f"{len(rows)} of {len(PRODUCTS)} products")

if __name__ == "__main__":
    app = QApplication(sys.argv)
    win = Inventory(); win.show()
    sys.exit(app.exec())

## Step 2 · Build it offscreen and take a screenshot
The same window, rendered to a PNG instead of your screen.

In [ ]:
import importlib, qt_inventory
from PySide6.QtWidgets import QApplication
importlib.reload(qt_inventory)
app = QApplication.instance() or QApplication([])
win = qt_inventory.Inventory()
win.show(); app.processEvents()
win.grab().save("qt_inventory.png")
def show_png(path, width=None):
    from IPython.display import Image, display
    display(Image(path, width=width))
show_png("qt_inventory.png", width=620)

## Step 3 · Simulate a user
Type into the search box and tick the checkbox from code, the way automated tests do.

In [ ]:
win.search.setText("mat"); app.processEvents()
print("Search 'mat':", [win.table.item(r, 0).text() for r in range(win.table.rowCount())], "|", win.statusBar().currentMessage())
win.search.clear(); win.low.setChecked(True); app.processEvents()
print("Low stock:", [win.table.item(r, 0).text() for r in range(win.table.rowCount())])
win.grab().save("qt_low_stock.png")
show_png("qt_low_stock.png", width=620)

## Step 4 · Run the real window on your laptop
Download `qt_inventory.py`, then run the commands in the last section of this notebook.

In [ ]:
import py_compile
py_compile.compile("qt_inventory.py", doraise=True)
print("qt_inventory.py is valid. Laptop: pip install pyside6, then python qt_inventory.py")

## ✍️ Your turn
Untick low stock, set the search text to `'o'`, process events, and store the **number of rows** in the table in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    exp = len([p for p in qt_inventory.PRODUCTS if "o" in p[0].lower()])
    return answer == exp, f"Expected {exp}, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
win.low.setChecked(False)
win.search.setText("o"); app.processEvents()
answer = win.table.rowCount()

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **QApplication**
- **QWidget / QMainWindow**
- **Layout**
- **Signal / slot**
- **QTableWidget**
- **Offscreen rendering**

**Next:** 61. Pygame Games (`pygame`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/61_pygame.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install pyside6
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.
4. **Run the app:** download `qt_inventory.py` (Colab: Files panel on the left → right-click → Download) into the `python-lab` folder, then run `python qt_inventory.py` in the same terminal.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.